<a href="https://colab.research.google.com/github/asdiFlv3/PHAS0056_assignments/blob/main/mini_project/prediction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### Loading, configs, parses and sanity checks

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import pandas as pd
import os

In [ ]:
INPUT_PATH = '/content/drive/MyDrive/sentiment_analysis_data/phase2_annotated_data/snippets_1919_2024_LF_20240628_v3.csv'
# only load the columns needed for prediction and analysis
cols_to_use = ["snippet", "date", "year", "current_party", "speakername"]

df = pd.read_csv(
    INPUT_PATH,
    usecols=cols_to_use,
    low_memory=False
)

# Drop rows where snippet is missing — nothing to predict on
df = df.dropna(subset=["snippet"])

print(f"Total snippets to process: {len(df)}")
print(df.head())

Total snippets to process: 262654
        speakername        date  year current_party  \
0     Mr. GRIFFITHS  1936-05-26  1936           NaN   
1       Mr. JENKINS  1936-05-26  1936           NaN   
2  Mr. J. GRIFFITHS  1936-05-26  1936           NaN   
3         Mr. VIANT  1936-05-26  1936           NaN   
4     Mr. ORR-EWING  1936-05-26  1936           NaN   

                                             snippet  
0  In view of the fact that these selling schemes...  
1  asked the Secretary for Mines whether the prop...  
2  In view of the fact that this committee was al...  
3  asked the Chancellor of the Exchequer whether ...  
4  I find myself in some agreement with much that...  


In [ ]:
# download dependencies
! pip install spacy[transformers] spacy-transformers
! python -m spacy download en_core_web_trf

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 795.8/795.8 kB 51.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 313.4/313.4 kB 33.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.0/10.0 MB 146.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 50.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 120.7 MB/s eta 0:00:00
  Attempting uninstall: huggingface-hub
    Found existing installation: huggingface_hub 1.6.0
    Uninstalling huggingface_hub-1.6.0:
      Successfully uninstalled huggingface_hub-1.6.0
  Attempting uninstall: tokenizers
    Found existing installation: tokenizers 0.22.2
    Uninstalling tokenizers-0.22.2:
      Successfully uninstalled tokenizers-0.22.2
  Attempting uninstall: transformers
    Found existing installation: transformers 5.0.0
    Uninstalling transformers-5.0.0:
      Successfully uninstalled tra

## Run predictions on the 2 model

We would like to save as predict because the prediction takes toooo long, once runtime reconnects we have to do it again, wasting both time and computational unit (money).

Separate the data into chunks with checkpoints appears every 5000 snippets (takes 5 mins on L4). Once training terminates accidentally, restarting the pipeline loads the progress csv, check the checkpoints ans start from the latest checkpoint.

In [ ]:
import spacy
from tqdm.auto import tqdm

# configs
CHECKPOINT_EVERY = 5000
BATCH_SIZE = 256
OUTPUT_DIR = "/content/drive/MyDrive/immigration_models/"
REL_PROGRESS_PATH = os.path.join(OUTPUT_DIR, "relevance_progress.csv")
SENT_PROGRESS_PATH = os.path.join(OUTPUT_DIR, "sentiment_progress.csv")

# load trained models from Drive
nlp_rel = spacy.load("/content/drive/MyDrive/immigration_models/relevance_output/model-best")
nlp_sent = spacy.load("/content/drive/MyDrive/immigration_models/sentiment_output/model-best")

# check checkpoints
if os.path.exists(REL_PROGRESS_PATH):
    done_df = pd.read_csv(REL_PROGRESS_PATH)
    start_from = len(done_df)
    print(f"Resuming from row {start_from}")
else:
    done_df = pd.DataFrame()
    start_from = 0
    print("Starting fresh")

# load data
df = pd.read_csv(INPUT_PATH, usecols=cols_to_use, low_memory=False)
df = df.dropna(subset=["snippet"]).reset_index(drop=True)

remaining = df.iloc[start_from:].copy()
print(f"Rows remaining: {len(remaining)}")

# initialize a list to hold the processed data chunks
results = []

# process the remaining rows in chunks to manage memory and save progress periodically
for chunk_start in range(0, len(remaining), CHECKPOINT_EVERY):
    # determine the end index for the current chunk
    chunk_end = min(chunk_start + CHECKPOINT_EVERY, len(remaining))
    chunk = remaining.iloc[chunk_start:chunk_end].copy()

    # extract the text data to feed into the model
    texts = chunk["snippet"].tolist()
    rel_preds = []

    # run the texts through the spacy model pipeline in batches
    # tqdm provides a progress bar for the current chunk
    for doc in tqdm(nlp_rel.pipe(texts, batch_size=BATCH_SIZE),
                    total=len(texts), desc="Relevance"):
        # the model returns multiple categories with probabilities; pick the highest one
        rel_preds.append(max(doc.cats, key=doc.cats.get))

    # assign the predictions back to the chunk DataFrame
    chunk["relevance_pred"] = rel_preds

    # store the processed chunk
    results.append(chunk)

    # checkpoint: combine previously done data with all newly processed chunks and save to disk
    checkpoint_df = pd.concat([done_df] + results, ignore_index=True)
    checkpoint_df.to_csv(REL_PROGRESS_PATH, index=False)

    # print progress update including the count of relevant snippets found
    processed_total = start_from + chunk_end
    imm_count = (chunk["relevance_pred"] == "IMMIGRATION").sum()
    print(f"Saved: {processed_total}/{len(df)} rows "
          f"({imm_count} immigration in this chunk)")

# final summary once all chunks are processed
print("\nRelevance prediction complete!")
print(f"Total: {len(checkpoint_df)}")
print(checkpoint_df["relevance_pred"].value_counts())


/tmp/ipykernel_7513/1028145069.py:17: DtypeWarning: Columns (3) have mixed types. Specify dtype option on import or set low_memory=False.
  done_df = pd.read_csv(REL_PROGRESS_PATH)


Resuming from row 140000
Rows remaining: 122654


Relevance:   0%|          | 0/5000 [00:00<?, ?it/s]

Saved: 145000/262654 rows (2719 immigration in this chunk)


Relevance:   0%|          | 0/5000 [00:00<?, ?it/s]

...

Saved: 255000/262654 rows (3336 immigration in this chunk)


Relevance:   0%|          | 0/5000 [00:00<?, ?it/s]

Saved: 260000/262654 rows (2940 immigration in this chunk)


Relevance:   0%|          | 0/2654 [00:00<?, ?it/s]

Saved: 262654/262654 rows (1324 immigration in this chunk)

Relevance prediction complete!
Total: 262654
relevance_pred
IMMIGRATION        137420
NOT_IMMIGRATION    125234
Name: count, dtype: int64


In [ ]:
# run sentiment model only on immigration-relevant snippets
# load relevance results, keep only immigration rows
rel_df = pd.read_csv(REL_PROGRESS_PATH)
imm_df = rel_df[rel_df["relevance_pred"] == "IMMIGRATION"].reset_index(drop=True)
print(f"Total immigration-relevant rows: {len(imm_df)}")

# resume check
if os.path.exists(SENT_PROGRESS_PATH):
    done_df = pd.read_csv(SENT_PROGRESS_PATH)
    start_from = len(done_df)
    print(f"Resuming from row {start_from}")
else:
    done_df = pd.DataFrame()
    start_from = 0
    print("Starting fresh")

remaining_sent = imm_df.iloc[start_from:].copy()
print(f"Rows remaining: {len(remaining)}")


Total immigration-relevant rows: 137420
Starting fresh
Rows remaining: 122654


In [ ]:
# initialize a list to hold the processed data chunks
results = []

# process the remaining rows in chunks to manage memory and save progress periodically
for chunk_start in range(0, len(remaining_sent), CHECKPOINT_EVERY):
    # determine the end index for the current chunk
    chunk_end = min(chunk_start + CHECKPOINT_EVERY, len(remaining_sent))
    chunk = remaining_sent.iloc[chunk_start:chunk_end].copy()

    # Extract the text data to feed into the model
    texts = chunk["snippet"].tolist()
    sent_preds = []

    # run the texts through the spacy model pipeline in batches
    # tqdm provides a progress bar for the current chunk
    for doc in tqdm(nlp_sent.pipe(texts, batch_size=BATCH_SIZE),
                    total=len(texts), desc="Sentiment"):
        # the model returns multiple categories with probabilities; pick the highest one
        sent_preds.append(max(doc.cats, key=doc.cats.get))

    # assign the predictions back to the chunk DataFrame
    chunk["sentiment_pred"] = sent_preds

    # store the processed chunk
    results.append(chunk)

    # checkpoint: combine previously done data with all newly processed chunks and save to disk
    checkpoint_df = pd.concat([done_df] + results, ignore_index=True)
    checkpoint_df.to_csv(SENT_PROGRESS_PATH, index=False)

    # print progress update
    processed_total = start_from + chunk_end
    print(f"Saved: {processed_total}/{len(imm_df)} rows")

# final summary once all chunks are processed
print("\nSentiment prediction complete!")
print(checkpoint_df["sentiment_pred"].value_counts())


Sentiment:   0%|          | 0/5000 [00:00<?, ?it/s]

Saved: 5000/137420 rows


Sentiment:   0%|          | 0/5000 [00:00<?, ?it/s]

Saved: 10000/137420 rows


Sentiment:   0%|          | 0/5000 [00:00<?, ?it/s]

...

Saved: 130000/137420 rows


Sentiment:   0%|          | 0/5000 [00:00<?, ?it/s]

Saved: 135000/137420 rows


Sentiment:   0%|          | 0/2420 [00:00<?, ?it/s]

Saved: 137420/137420 rows

Sentiment prediction complete!
sentiment_pred
NEUTRAL     78216
POSITIVE    37063
NEGATIVE    22141
Name: count, dtype: int64
